# Demo 3: Complex SQL Queries

In this notebook, we'll explore more advanced SQL concepts using our NHANES data. We'll cover:

1. Common Table Expressions (CTEs)
2. Window Functions
3. Advanced JOIN Operations
4. Subqueries
5. Performance Optimization

## Setup

In [1]:
%pip install jupysql duckdb-engine pandas polars --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sqlalchemy import create_engine
%load_ext sql

# Configure SQL magic for better output
%config SqlMagic.autopandas = True
%config SqlMagic.autocommit=True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Connect to DuckDB
# Execute SQL queries with pandas' `read_sql` function
# results_df = pd.read_sql("SELECT * FROM demographics LIMIT 5", engine)
engine = create_engine('duckdb:///:memory:')

# Use this engine for jupysql
%sql engine

# NOTE: memory db's are not shared between connections
# If using only jupysql, you can use the following:
# %sql duckdb:///:memory:

## Load Data

Let's load our NHANES data directly from CSV files:

In [3]:
%%sql
-- Clean up any existing tables
DROP TABLE IF EXISTS questionnaire;
DROP TABLE IF EXISTS laboratory;
DROP TABLE IF EXISTS examination;
DROP TABLE IF EXISTS demographics;

-- Load demographics
CREATE TABLE demographics AS
SELECT * FROM read_csv_auto('data/demographics.csv');

-- Load examination
CREATE TABLE examination AS
SELECT * FROM read_csv_auto('data/examination.csv');

-- Load laboratory
CREATE TABLE laboratory AS
SELECT * FROM read_csv_auto('data/labs.csv');

-- Load questionnaire
CREATE TABLE questionnaire AS
SELECT * FROM read_csv_auto('data/questionnaire.csv');

,Success


## Common Table Expressions (CTEs)

CTEs help make complex queries more readable and maintainable:

In [4]:
%%sql
-- Calculate BMI categories using CTE
WITH bmi_categories AS (
    SELECT 
        d.SEQN,
        d.RIDAGEYR,
        d.RIAGENDR,
        e.BMXBMI,
        CASE
            WHEN e.BMXBMI < 18.5 THEN 'Underweight'
            WHEN e.BMXBMI < 25 THEN 'Normal'
            WHEN e.BMXBMI < 30 THEN 'Overweight'
            ELSE 'Obese'
        END AS bmi_category
    FROM demographics d
    JOIN examination e ON d.SEQN = e.SEQN
)
SELECT 
    RIAGENDR,
    bmi_category,
    COUNT(*) AS count,
    AVG(RIDAGEYR) AS avg_age
FROM bmi_categories
GROUP BY RIAGENDR, bmi_category
ORDER BY RIAGENDR, bmi_category;

,RIAGENDR,bmi_category,count,avg_age
0,1,Normal,1341,30.932140
1,1,Obese,1381,33.729182
2,1,Overweight,1166,45.391938
3,1,Underweight,943,8.184517
4,2,Normal,1461,32.253936
5,2,Obese,1724,37.868329
6,2,Overweight,933,44.505895
7,2,Underweight,864,8.962963


## Window Functions

Window functions allow us to perform calculations across rows related to the current row:

In [5]:
%%sql
-- Calculate running average of BMI by age
SELECT 
    SEQN,
    RIDAGEYR,
    RIAGENDR,
    BMXBMI,
    AVG(BMXBMI) OVER (PARTITION BY RIAGENDR ORDER BY RIDAGEYR) AS running_avg_bmi
FROM (
    SELECT d.SEQN, d.RIDAGEYR, d.RIAGENDR, e.BMXBMI
    FROM demographics d
    JOIN examination e ON d.SEQN = e.SEQN
) AS combined
ORDER BY RIAGENDR, RIDAGEYR;

,SEQN,RIDAGEYR,RIAGENDR,BMXBMI,running_avg_bmi
0,77925,0,1,NaN,NaN
1,78011,0,1,NaN,NaN
2,78023,0,1,NaN,NaN
3,78102,0,1,NaN,NaN
4,78123,0,1,NaN,NaN
...,...,...,...,...,...
9808,83600,80,2,28.0,26.247478
9809,83630,80,2,26.0,26.247478
9810,83639,80,2,14.2,26.247478
9811,83702,80,2,27.9,26.247478


## Advanced JOIN Operations

Let's explore different types of JOINs and their use cases:

In [6]:
%%sql
-- INNER JOIN: Get participants with complete data
SELECT d.SEQN, d.RIDAGEYR, d.RIAGENDR, e.BMXBMI, l.LBXTC
FROM demographics d
INNER JOIN examination e ON d.SEQN = e.SEQN
INNER JOIN laboratory l ON d.SEQN = l.SEQN
WHERE e.BMXBMI IS NOT NULL AND l.LBXTC IS NOT NULL
LIMIT 10;

,SEQN,RIDAGEYR,RIAGENDR,BMXBMI,LBXTC
0,73557,69,1,26.7,167
1,73558,54,1,28.6,170
2,73559,72,1,28.9,126
3,73560,9,1,17.1,168
4,73561,73,2,19.7,201
5,73562,56,1,41.7,226
6,73564,61,2,35.7,168
7,73566,56,2,26.5,278
8,73567,65,1,22.0,173
9,73568,26,2,20.3,168


In [7]:
%%sql
-- LEFT JOIN: Get all participants with available data
SELECT d.SEQN, d.RIDAGEYR, d.RIAGENDR, e.BMXBMI, l.LBXTC
FROM demographics d
LEFT JOIN examination e ON d.SEQN = e.SEQN
LEFT JOIN laboratory l ON d.SEQN = l.SEQN
LIMIT 10;

,SEQN,RIDAGEYR,RIAGENDR,BMXBMI,LBXTC
0,73557,69,1,26.7,167
1,73558,54,1,28.6,170
2,73559,72,1,28.9,126
3,73560,9,1,17.1,168
4,73561,73,2,19.7,201
5,73562,56,1,41.7,226
6,73563,0,1,NaN,<NA>
7,73564,61,2,35.7,168
8,73566,56,2,26.5,278
9,73567,65,1,22.0,173


## Subqueries

Subqueries help break down complex problems:

In [8]:
%%sql
-- Find participants with above average BMI
SELECT SEQN, RIDAGEYR, RIAGENDR, BMXBMI
FROM (
    SELECT d.SEQN, d.RIDAGEYR, d.RIAGENDR, e.BMXBMI
    FROM demographics d
    JOIN examination e ON d.SEQN = e.SEQN
) AS combined
WHERE BMXBMI > (SELECT AVG(BMXBMI) FROM examination WHERE BMXBMI IS NOT NULL)
ORDER BY BMXBMI DESC
LIMIT 10;

,SEQN,RIDAGEYR,RIAGENDR,BMXBMI
0,77055,45,2,82.9
1,75555,58,2,77.5
2,77077,50,1,74.1
3,81523,57,2,71.5
4,77239,36,2,70.1
5,74910,19,2,68.6
6,76109,36,1,67.9
7,82389,34,2,67.5
8,78239,47,2,65.5
9,73797,57,2,64.7


## Performance Optimization

Let's look at some performance optimization techniques:

In [9]:
%%sql
-- Use appropriate indexes
CREATE INDEX idx_demographics_seqn ON demographics(SEQN);
CREATE INDEX idx_examination_seqn ON examination(SEQN);
CREATE INDEX idx_laboratory_seqn ON laboratory(SEQN);

,Success


In [10]:
%%sql
-- Create a view for health summary
-- NOTE: DuckDB does not support materialized views
CREATE VIEW health_summary AS
SELECT 
    d.SEQN,
    d.RIDAGEYR,
    d.RIAGENDR,
    e.BMXBMI,
    l.LBXTC,
    q.DIQ010
FROM demographics d
LEFT JOIN examination e ON d.SEQN = e.SEQN
LEFT JOIN laboratory l ON d.SEQN = l.SEQN
LEFT JOIN questionnaire q ON d.SEQN = q.SEQN;

-- Query the view
-- NOTE: This is done on the fly by querying the underlying tables
SELECT * FROM health_summary LIMIT 5;

,SEQN,RIDAGEYR,RIAGENDR,BMXBMI,LBXTC,DIQ010
0,73557,69,1,26.7,167,1
1,73558,54,1,28.6,170,1
2,73559,72,1,28.9,126,1
3,73560,9,1,17.1,168,2
4,73561,73,2,19.7,201,2


## Practice

Try these exercises:
1. Create a CTE to analyze blood pressure trends by age group
2. Use window functions to find the top 10% of participants by BMI in each age group
3. Write a query to find participants with multiple health risk factors
4. Optimize a complex query using materialized views